# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [15]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.3 MB/s eta 0:00:00


In [16]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


## 1) Load dataset and convert to Documents


In [18]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
        page_content=row[text_column], metadata={"source": row[source_column]}
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])

Documents: 200
Example: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx'}
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 



## 2) Split into chunks


In [20]:
chunk_size = 500 # To-Do: specify chunk size, a way to estimate is to divide average document length by desired number of chunks
chunk_overlap = 50 # To-Do: specify chunk overlap, a way to estimate is to take a small percentage of chunk size

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap # To-Do: fill in chunk_size and chunk_overlap appropriately
)

splits = splitter.split_documents(documents)
print("Chunks:", len(splits))
print("First chunk:", splits[0].metadata)
print(splits[0].page_content[:350])

Chunks: 5966
First chunk: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx'}
Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 




## 3) Vector store + retriever (FAISS)


In [24]:
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

# Filtrer les documents avec un contexte non vide et ajouter une métadonnée "source" plus parlante
docs_all = [d for d in data if d.page_content and d.page_content.strip()]

for i, d in enumerate(docs_all):
    category = d.metadata.get("category", "unknown")
    d.metadata["source"] = f"dolly-15k:{category}:{i}"

print(f"Documents avec contexte non vide: {len(docs_all)} / {len(data)}")
print("Exemple de metadata enrichie:", docs_all[0].metadata)

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Documents avec contexte non vide: 15011 / 15011
Exemple de metadata enrichie: {'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa', 'source': 'dolly-15k:closed_qa:0'}


## 4) Build the RAG chain


In [35]:
doc_subset = docs_all[:1000]  # sous-ensemble pour limiter le temps d'indexation

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(doc_subset)

print("Config principale (1000/150):", len(docs), "chunks")
print(docs[0])

Config principale (1000/150): 1201 chunks
page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa', 'source': 'dolly-15k:closed_qa:0'}


In [36]:
text_splitter_small = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
docs_small = text_splitter_small.split_documents(doc_subset)

print("Config alternative (400/50):", len(docs_small), "chunks")
print(docs_small[0])

Config alternative (400/50): 1777 chunks
page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa', 'source': 'dolly-15k:closed_qa:0'}


In [38]:
from langchain_community.embeddings import HuggingFaceEmbeddings as _HFE

_embeddings_probe = _HFE(
    model_name="sentence-transformers/all-MiniLM-l6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": False},
)

db_1000 = FAISS.from_documents(docs, _embeddings_probe)
db_400 = FAISS.from_documents(docs_small, _embeddings_probe)

chunk_test_question = "What is cheesemaking?"

for label, store in [("1000/150", db_1000), ("400/50", db_400)]:
    print(f"\n=== Chunks récupérés avec la config {label} (k=4) ===")
    results = store.as_retriever(search_kwargs={"k": 4}).invoke(chunk_test_question)
    for i, res in enumerate(results):
        print(f"\n--- Chunk {i + 1} | source: {res.metadata.get('source')} | longueur: {len(res.page_content)} ---")
        print(res.page_content[:250])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


=== Chunks récupérés avec la config 1000/150 (k=4) ===

--- Chunk 1 | source: dolly-15k:summarization:407 | longueur: 900 ---
"A grilled cheese sandwich is made by placing a cheese filling, often cheddar or American cheese, between two slices of bread, which is then heated until the bread browns and the cheese melts. A layer of butter or mayonnaise may be added to the outsi

--- Chunk 2 | source: dolly-15k:closed_qa:200 | longueur: 846 ---
"Wine is an alcoholic drink typically made from fermented grapes. Yeast consumes the sugar in the grapes and converts it to ethanol and carbon dioxide, releasing heat in the process. Different varieties of grapes and strains of yeasts are major facto

--- Chunk 3 | source: dolly-15k:summarization:625 | longueur: 504 ---
sheep and goat producers indicate a special bond quickly develops between lambs and their guard llama and the llama is particularly protective of the lambs.\n\nUsing llamas as guards has reduced the losses to predators for many produ

## 5) Demo: RAG vs no-RAG

In [47]:
# Define a default retriever from db_1000
retriever = db_1000.as_retriever(search_kwargs={"k": 4})
print("Default retriever defined with k=4")

Default retriever defined with k=4


In [52]:
# 5. Build the Vector Store and Retriever

# 1. Install dependencies (if needed)
# pip install langchain faiss-cpu sentence-transformers

# 2. Imports
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 3. Load embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 4. Create FAISS vector stores (using both chunking strategies)
db_small = FAISS.from_documents(docs_small, embeddings)
db_large = FAISS.from_documents(docs_large, embeddings)

# 5. Build retrievers with different k values
retriever_k2 = db_small.as_retriever(search_kwargs={"k": 2})
retriever_k4 = db_small.as_retriever(search_kwargs={"k": 4})
retriever_k6 = db_small.as_retriever(search_kwargs={"k": 6})

# 6. Test query
query = "What is the main topic?"

results_k2 = retriever_k2.get_relevant_documents(query)
results_k4 = retriever_k4.get_relevant_documents(query)
results_k6 = retriever_k6.get_relevant_documents(query)

# 7. Display results
def print_results(results, title):
    print(f"\n--- {title} ---")
    for i, doc in enumerate(results):
        print(f"\nResult {i+1}:\n{doc.page_content[:300]}...")

print_results(results_k2, "Top 2 Results (k=2)")
print_results(results_k4, "Top 4 Results (k=4)")
print_results(results_k6, "Top 6 Results (k=6)")

# 8. (Optional) Compare with large chunks
retriever_large_k4 = db_large.as_retriever(search_kwargs={"k": 4})
results_large = retriever_large_k4.get_relevant_documents(query)

print_results(results_large, "Large Chunks (k=4)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

NameError: name 'docs_large' is not defined

In [54]:
pip install langchain langchain-community sentence-transformers faiss-cpu --break-system-packages

In [55]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},          # passe à "cuda" si GPU disponible
    encode_kwargs={"normalize_embeddings": True}  # utile pour la similarité cosinus
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [60]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document # Corrected import path

# Si tes chunks sont de simples strings, convertis-les d'abord
# The variable 'chunks' was not defined, using 'docs' which contains chunked documents
current_chunks = docs # Assuming 'docs' is the intended variable for chunks

if isinstance(current_chunks[0], str):
    current_chunks = [Document(page_content=chunk) for chunk in current_chunks]

vectorstore = FAISS.from_documents(current_chunks, embedding_model)

# Optionnel : sauvegarder l'index localement
vectorstore.save_local("faiss_index")

# Pour le recharger plus tard :
# vectorstore = FAISS.load_local("faiss_index", embedding_model, allow_dangerous_deserialization=True)

## 6) Retrieval sanity check

In [44]:
retrieval_question = "What are the benefits of using Databricks?"

print(f"\n=== Chunks retrieved for k=4 (default retriever) ===")
results_k4 = retriever.invoke(retrieval_question)
for i, res in enumerate(results_k4):
    print(f"\n--- Chunk {i + 1} | source: {res.metadata.get('source')} | length: {len(res.page_content)} ---")
    print(res.page_content[:300] + "...")


=== Chunks retrieved for k=4 (default retriever) ===

--- Chunk 1 | source: dolly-15k:information_extraction:563 | length: 414 ---
"Flink provides a high-throughput, low-latency streaming engine as well as support for event-time processing and state management. Flink applications are fault-tolerant in the event of machine failure and support exactly-once semantics. Programs can be written in Java, Scala, Python, and SQL and are...

--- Chunk 2 | source: dolly-15k:closed_qa:830 | length: 822 ---
"The technique was originally developed by Sakichi Toyoda and was used within the Toyota Motor Corporation during the evolution of its manufacturing methodologies. It is a critical component of problem-solving training, delivered as part of the induction into the Toyota Production System. The archit...

--- Chunk 3 | source: dolly-15k:summarization:36 | length: 1000 ---
"LinkedIn (/l\u026a\u014bkt\u02c8\u026an/) is a business and employment-focused social media platform that works through websi

In [45]:
# Experiment with k=2
retriever_k2 = db_1000.as_retriever(search_kwargs={"k": 2})
print(f"\n=== Chunks retrieved for k=2 ===")
results_k2 = retriever_k2.invoke(retrieval_question)
for i, res in enumerate(results_k2):
    print(f"\n--- Chunk {i + 1} | source: {res.metadata.get('source')} | length: {len(res.page_content)} ---")
    print(res.page_content[:300] + "...")


=== Chunks retrieved for k=2 ===

--- Chunk 1 | source: dolly-15k:information_extraction:563 | length: 414 ---
"Flink provides a high-throughput, low-latency streaming engine as well as support for event-time processing and state management. Flink applications are fault-tolerant in the event of machine failure and support exactly-once semantics. Programs can be written in Java, Scala, Python, and SQL and are...

--- Chunk 2 | source: dolly-15k:closed_qa:830 | length: 822 ---
"The technique was originally developed by Sakichi Toyoda and was used within the Toyota Motor Corporation during the evolution of its manufacturing methodologies. It is a critical component of problem-solving training, delivered as part of the induction into the Toyota Production System. The archit...


In [46]:
# Experiment with k=6
retriever_k6 = db_1000.as_retriever(search_kwargs={"k": 6})
print(f"\n=== Chunks retrieved for k=6 ===")
results_k6 = retriever_k6.invoke(retrieval_question)
for i, res in enumerate(results_k6):
    print(f"\n--- Chunk {i + 1} | source: {res.metadata.get('source')} | length: {len(res.page_content)} ---")
    print(res.page_content[:300] + "...")


=== Chunks retrieved for k=6 ===

--- Chunk 1 | source: dolly-15k:information_extraction:563 | length: 414 ---
"Flink provides a high-throughput, low-latency streaming engine as well as support for event-time processing and state management. Flink applications are fault-tolerant in the event of machine failure and support exactly-once semantics. Programs can be written in Java, Scala, Python, and SQL and are...

--- Chunk 2 | source: dolly-15k:closed_qa:830 | length: 822 ---
"The technique was originally developed by Sakichi Toyoda and was used within the Toyota Motor Corporation during the evolution of its manufacturing methodologies. It is a critical component of problem-solving training, delivered as part of the induction into the Toyota Production System. The archit...

--- Chunk 3 | source: dolly-15k:summarization:36 | length: 1000 ---
"LinkedIn (/l\u026a\u014bkt\u02c8\u026an/) is a business and employment-focused social media platform that works through websites and mobile apps.

In [59]:
def test_k_values(query, k_values=[2, 4, 6]):
    for k in k_values:
        retriever_k = vectorstore.as_retriever(search_kwargs={"k": k})
        results = retriever_k.invoke(query)
        print(f"\n{'='*50}")
        print(f"k = {k} → {len(results)} documents récupérés")
        print(f"{'='*50}")
        for i, doc in enumerate(results):
            preview = doc.page_content[:150].replace("\n", " ")
            print(f"  [{i+1}] {preview}...")

test_k_values("Ta question de test ici")


k = 2 → 2 documents récupérés
  [1] "System and Organization Controls (SOC), (also sometimes referred to as service organizations controls) as defined by the American Institute of Certif...
  [2] the design of specified controls meet the relevant trust principles. (Are the design and documentation likely to accomplish the goals defined in the r...

k = 4 → 4 documents récupérés
  [1] "System and Organization Controls (SOC), (also sometimes referred to as service organizations controls) as defined by the American Institute of Certif...
  [2] the design of specified controls meet the relevant trust principles. (Are the design and documentation likely to accomplish the goals defined in the r...
  [3] of reporting, type 1 and type 2. Additional AICPA guidance materials specify three types of reporting: SOC 1, SOC 2, and SOC 3.\n\nTrust Service Princ...
  [4] the past few years, which has led to a steady deterioration in its financial position. This has resulted in potential loan losses, w

7. RAG question answering

In [61]:
pip install transformers langchain langchain-community --break-system-packages

In [63]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

hf_pipeline = pipeline(
    "text-generation", # Changed from "text2text-generation" to "text-generation"
    model="google/flan-t5-small",
    max_new_tokens=256,
    temperature=0.1,      # peu de créativité, on veut qu'il colle au contexte
    do_sample=False       # génération déterministe
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertL

In [65]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",              # empile les chunks récupérés dans le prompt
    retriever=retriever,             # k=4 par défaut
    return_source_documents=True     # indispensable pour vérifier les sources
)

In [66]:
questions = [
    "Question 1 ici",
    "Question 2 ici",
    "Question 3 ici",
]

for q in questions:
    result = qa_chain.invoke({"query": q})

    print("=" * 60)
    print(f"QUESTION : {q}")
    print("-" * 60)
    print(f"RÉPONSE  : {result['result']}")
    print("-" * 60)
    print("SOURCES RÉCUPÉRÉES :")
    for i, doc in enumerate(result["source_documents"]):
        preview = doc.page_content[:200].replace("\n", " ")
        metadata = doc.metadata  # ex: source, page, chunk_id...
        print(f"  [{i+1}] {preview}...")
        print(f"      metadata: {metadata}")
    print()

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (932 > 512). Running this sequence through the model will result in indexing errors
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION : Question 1 ici
------------------------------------------------------------
RÉPONSE  : Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

"The potential uses of AI in government are wide and varied, with Deloitte considering that \"Cognitive technologies could eventually revolutionize every facet of government operations\". Mehr suggests that six types of government problems are appropriate for AI applications:\n- Resource allocation - such as where administrative support is required to complete tasks more quickly.\n- Large datasets - where these are too large for employees to work efficiently and multiple datasets could be combined to provide greater insights.\n- Experts shortage - including where basic questions could be answered and niche issues can be learned.\n- Predictable scenario - historical data makes the situation predictable.\n- Procedural - repetitive

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION : Question 2 ici
------------------------------------------------------------
RÉPONSE  : Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

"The potential uses of AI in government are wide and varied, with Deloitte considering that \"Cognitive technologies could eventually revolutionize every facet of government operations\". Mehr suggests that six types of government problems are appropriate for AI applications:\n- Resource allocation - such as where administrative support is required to complete tasks more quickly.\n- Large datasets - where these are too large for employees to work efficiently and multiple datasets could be combined to provide greater insights.\n- Experts shortage - including where basic questions could be answered and niche issues can be learned.\n- Predictable scenario - historical data makes the situation predictable.\n- Procedural - repetitive

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION : Question 3 ici
------------------------------------------------------------
RÉPONSE  : Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

"The potential uses of AI in government are wide and varied, with Deloitte considering that \"Cognitive technologies could eventually revolutionize every facet of government operations\". Mehr suggests that six types of government problems are appropriate for AI applications:\n- Resource allocation - such as where administrative support is required to complete tasks more quickly.\n- Large datasets - where these are too large for employees to work efficiently and multiple datasets could be combined to provide greater insights.\n- Experts shortage - including where basic questions could be answered and niche issues can be learned.\n- Predictable scenario - historical data makes the situation predictable.\n- Procedural - repetitive

In [67]:
def evaluate_grounding(question, answer, source_documents):
    """Vérifie si des mots-clés de la réponse apparaissent dans les sources."""
    combined_sources = " ".join(doc.page_content.lower() for doc in source_documents)
    answer_words = set(answer.lower().split())
    overlap = [w for w in answer_words if len(w) > 4 and w in combined_sources]
    ratio = len(overlap) / max(len(answer_words), 1)
    print(f"Chevauchement réponse/sources : {ratio:.0%} ({len(overlap)} mots-clés trouvés)")
    return ratio

for q in questions:
    result = qa_chain.invoke({"query": q})
    print(f"\nQ: {q}")
    print(f"R: {result['result']}")
    evaluate_grounding(q, result['result'], result['source_documents'])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Question 1 ici
R: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

"The potential uses of AI in government are wide and varied, with Deloitte considering that \"Cognitive technologies could eventually revolutionize every facet of government operations\". Mehr suggests that six types of government problems are appropriate for AI applications:\n- Resource allocation - such as where administrative support is required to complete tasks more quickly.\n- Large datasets - where these are too large for employees to work efficiently and multiple datasets could be combined to provide greater insights.\n- Experts shortage - including where basic questions could be answered and niche issues can be learned.\n- Predictable scenario - historical data makes the situation predictable.\n- Procedural - repetitive tasks where inputs or outputs have a binary answer.\n- Diverse data - where

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Question 2 ici
R: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

"The potential uses of AI in government are wide and varied, with Deloitte considering that \"Cognitive technologies could eventually revolutionize every facet of government operations\". Mehr suggests that six types of government problems are appropriate for AI applications:\n- Resource allocation - such as where administrative support is required to complete tasks more quickly.\n- Large datasets - where these are too large for employees to work efficiently and multiple datasets could be combined to provide greater insights.\n- Experts shortage - including where basic questions could be answered and niche issues can be learned.\n- Predictable scenario - historical data makes the situation predictable.\n- Procedural - repetitive tasks where inputs or outputs have a binary answer.\n- Diverse data - where

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Question 3 ici
R: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

"The potential uses of AI in government are wide and varied, with Deloitte considering that \"Cognitive technologies could eventually revolutionize every facet of government operations\". Mehr suggests that six types of government problems are appropriate for AI applications:\n- Resource allocation - such as where administrative support is required to complete tasks more quickly.\n- Large datasets - where these are too large for employees to work efficiently and multiple datasets could be combined to provide greater insights.\n- Experts shortage - including where basic questions could be answered and niche issues can be learned.\n- Predictable scenario - historical data makes the situation predictable.\n- Procedural - repetitive tasks where inputs or outputs have a binary answer.\n- Diverse data - where